# PCE inflation breadth

Calculates monthly unweighted and current-dollar-PCE-weighted inflation breadth for the categories in `Detailed PCE.xlsx`, beginning in 2000, and decomposes both measures by Major Group. For lookbacks shorter than 12 months, inflation is annualized.

In [ ]:
# User inputs
API_KEY = "FD4E7F46-5D89-4CA8-BE48-4F5997F83B3B"
THRESHOLD = 3.0       # percent
LOOKBACK_MONTHS = 12  # for example: 12, 6, or 3
START_YEAR = 2000

CATEGORIES_FILE = "Detailed PCE.xlsx"

In [ ]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd

if LOOKBACK_MONTHS <= 0:
    raise ValueError("LOOKBACK_MONTHS must be positive.")

categories = pd.read_excel(CATEGORIES_FILE)
categories["Line Item"] = pd.to_numeric(categories["Line Item"]).astype(int)

def get_bea_table(table_name):
    current_year = pd.Timestamp.today().year
    years_of_history = (LOOKBACK_MONTHS + 11) // 12
    first_year = START_YEAR - years_of_history
    params = {
        "UserID": API_KEY,
        "method": "GetData",
        "DataSetName": "NIUnderlyingDetail",
        "TableName": table_name,
        "Frequency": "M",
        "Year": ",".join(map(str, range(first_year, current_year + 1))),
        "ResultFormat": "JSON",
    }
    url = "https://apps.bea.gov/api/data/?" + urlencode(params)
    with urlopen(url, timeout=60) as response:
        payload = json.load(response)

    results = payload["BEAAPI"]["Results"]
    if "Data" not in results:
        raise RuntimeError(f"BEA API error: {results}")

    data = pd.DataFrame(results["Data"])
    data["Line Item"] = pd.to_numeric(data["LineNumber"], errors="coerce")
    data["Date"] = pd.PeriodIndex(
        data["TimePeriod"].str.replace("M", "-", regex=False), freq="M"
    )
    values = (
        data["DataValue"].astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("(", "-", regex=False)
        .str.replace(")", "", regex=False)
    )
    data["Value"] = pd.to_numeric(values, errors="coerce")
    return data[["Line Item", "Date", "Value"]]

prices = get_bea_table("U20404")   # Detailed PCE price indexes
spending = get_bea_table("U20405") # Detailed current-dollar PCE

line_items = categories["Line Item"].tolist()
price_panel = (
    prices[prices["Line Item"].isin(line_items)]
    .pivot(index="Date", columns="Line Item", values="Value")
    .sort_index()
)
spending_panel = (
    spending[spending["Line Item"].isin(line_items)]
    .pivot(index="Date", columns="Line Item", values="Value")
    .sort_index()
)

inflation_panel = (
    (price_panel / price_panel.shift(LOOKBACK_MONTHS))
    ** (12 / LOOKBACK_MONTHS) - 1
) * 100

start_period = pd.Period(f"{START_YEAR}-01", freq="M")
dates = inflation_panel.index.intersection(spending_panel.index)
dates = dates[dates >= start_period]
inflation = inflation_panel.reindex(dates)
pce = spending_panel.reindex(index=dates, columns=inflation.columns)

price_valid = inflation.notna()
above = inflation.gt(THRESHOLD) & price_valid
valid_count = price_valid.sum(axis=1)
above_count = above.sum(axis=1)
unweighted_share = above_count.div(valid_count).where(valid_count.gt(0))

weighted_valid = price_valid & pce.notna()
weighted_numerator = pce.where(above & weighted_valid, 0.0).sum(axis=1)
weighted_denominator = pce.where(weighted_valid).sum(axis=1, min_count=1)
weighted_share = weighted_numerator.div(weighted_denominator)

breadth_time_series = pd.DataFrame({
    "Categories above": above_count,
    "Valid categories": valid_count,
    "Unweighted share above threshold": unweighted_share,
    "PCE-weighted share above threshold": weighted_share,
}).dropna(how="all")
breadth_time_series.index.name = "Date"

print(
    f"{len(breadth_time_series)} monthly observations: "
    f"{breadth_time_series.index.min()} to {breadth_time_series.index.max()}"
)

## 1. Unweighted share

Each category with a valid inflation rate receives the same weight. The complete history is stored in `unweighted_time_series`; the table displays the latest 12 months.

In [ ]:
unweighted_time_series = breadth_time_series[[
    "Categories above",
    "Valid categories",
    "Unweighted share above threshold",
]]
print(unweighted_time_series.tail(12).to_string(formatters={
    "Unweighted share above threshold": "{:.1%}".format,
}))

## 2. PCE-weighted share

Each month's weights are that month's current-dollar PCE values from table 2.4.5U, normalized across categories with valid inflation and spending data. The complete history is stored in `weighted_time_series`; the table displays the latest 12 months.

In [ ]:
weighted_time_series = breadth_time_series[[
    "PCE-weighted share above threshold",
]]
print(weighted_time_series.tail(12).to_string(formatters={
    "PCE-weighted share above threshold": "{:.1%}".format,
}))

## 3. Contributions by Major Group

Each contribution uses the denominator of the corresponding overall breadth measure. Therefore, the Major Group contributions sum exactly to the unweighted and PCE-weighted shares above the threshold. The complete histories are stored in `unweighted_group_contributions` and `weighted_group_contributions`.

In [ ]:
group_order = categories["Major Group"].dropna().drop_duplicates().tolist()
line_to_group = categories.set_index("Line Item")["Major Group"]
column_groups = line_to_group.reindex(inflation.columns)

unweighted_group_contributions = pd.DataFrame(index=dates)
weighted_group_contributions = pd.DataFrame(index=dates)

for group in group_order:
    group_columns = column_groups.index[column_groups.eq(group)]

    group_above_count = above[group_columns].sum(axis=1)
    unweighted_group_contributions[group] = group_above_count.div(valid_count)

    group_weighted_numerator = pce[group_columns].where(
        above[group_columns] & weighted_valid[group_columns], 0.0
    ).sum(axis=1)
    weighted_group_contributions[group] = (
        group_weighted_numerator.div(weighted_denominator)
    )

unweighted_group_contributions.index.name = "Date"
weighted_group_contributions.index.name = "Date"

unweighted_difference = (
    unweighted_group_contributions.sum(axis=1) - unweighted_share
).abs().max()
weighted_difference = (
    weighted_group_contributions.sum(axis=1) - weighted_share
).abs().max()
assert unweighted_difference < 1e-10
assert weighted_difference < 1e-10

latest_date = breadth_time_series.index.max()
latest_major_group_contributions = pd.DataFrame({
    "Unweighted contribution": unweighted_group_contributions.loc[latest_date],
    "PCE-weighted contribution": weighted_group_contributions.loc[latest_date],
})
latest_major_group_contributions.loc["Total"] = (
    latest_major_group_contributions.sum(axis=0)
)

print(f"Major Group contributions for {latest_date}:")
print(latest_major_group_contributions.to_string(formatters={
    "Unweighted contribution": "{:.2%}".format,
    "PCE-weighted contribution": "{:.2%}".format,
}))

In [ ]:
# Optional stacked time-series charts (requires matplotlib)
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Install matplotlib to display the stacked contribution charts.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    (unweighted_group_contributions * 100).plot.area(
        ax=axes[0], stacked=True, linewidth=0, legend=False
    )
    axes[0].set_title("Unweighted contributions")
    axes[0].set_ylabel("Percentage points")
    axes[0].set_ylim(0, 100)

    (weighted_group_contributions * 100).plot.area(
        ax=axes[1], stacked=True, linewidth=0
    )
    axes[1].set_title("PCE-weighted contributions")
    axes[1].set_ylabel("Percentage points")
    axes[1].set_xlabel("Date")
    axes[1].set_ylim(0, 100)
    axes[1].legend(
        title="Major Group", bbox_to_anchor=(1.02, 1), loc="upper left"
    )

    fig.suptitle(
        f"PCE inflation breadth above {THRESHOLD:.1f}% "
        f"({LOOKBACK_MONTHS}-month measure)"
    )
    plt.tight_layout()
    plt.show()